<a href="https://colab.research.google.com/github/divyadharshiniug/24ADI006L_24BAD023/blob/main/24BAD023_DL_EX_9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# EXPT 9 - Transformer-based LLM Chatbot using Gradio
# PART A: LOADING AND CONFIGURING A PRE-TRAINED LLM
# Name    : Divyadharshini U G
# Roll No : 24BAD023
# ============================================================

!pip install -q transformers accelerate gradio

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

print("Name    : Divyadharshini U G")
print("Roll No : 24BAD023")
print("-" * 50)

# 1. Select a pre-trained LLM from Hugging Face
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Model  :", MODEL_NAME)
print("Device :", device)

# 2. Load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
).to(device)
model.eval()

total_params = sum(p.numel() for p in model.parameters())
print("Total parameters :", f"{total_params:,}")
print("Vocabulary size  :", tokenizer.vocab_size)
print("-" * 50)

# 3. Prepare and tokenize a sample user prompt
sample_prompt = "What is machine learning?"
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": sample_prompt},
]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors="pt").to(device)

print("User prompt        :", sample_prompt)
print("Number of tokens   :", inputs["input_ids"].shape[1])
print("First 15 token IDs :", inputs["input_ids"][0][:15].tolist())
print("First 15 tokens    :", tokenizer.convert_ids_to_tokens(inputs["input_ids"][0][:15]))
print("-" * 50)

# 4. Generate a response with configured parameters
TEMPERATURE = 0.7       # controls randomness
MAX_NEW_TOKENS = 120    # maximum output length

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        temperature=TEMPERATURE,
        do_sample=True,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id,
    )

# Keep only the newly generated tokens (skip the prompt)
new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
response = tokenizer.decode(new_tokens, skip_special_tokens=True)

# 5. Display and examine the response
print("Temperature        :", TEMPERATURE)
print("Max output tokens  :", MAX_NEW_TOKENS)
print("Tokens generated   :", len(new_tokens))
print("-" * 50)
print("LLM RESPONSE:\n")
print(response)

# 6. Examine the effect of temperature on the same prompt
print("\n" + "=" * 50)
print("EFFECT OF TEMPERATURE (same prompt)")
print("=" * 50)
for temp in [0.2, 1.0]:
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=60,
            temperature=temp,
            do_sample=True,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )
    reply = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"\nTemperature = {temp}:\n{reply}")

Name    : Divyadharshini U G
Roll No : 24BAD023
--------------------------------------------------
Model  : Qwen/Qwen2.5-0.5B-Instruct
Device : cpu


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

... (2 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Total parameters : 494,032,768
Vocabulary size  : 151643
--------------------------------------------------
User prompt        : What is machine learning?
Number of tokens   : 24
First 15 token IDs : [151644, 8948, 198, 2610, 525, 264, 10950, 17847, 13, 151645, 198, 151644, 872, 198, 3838]
First 15 tokens    : ['<|im_start|>', 'system', 'Ċ', 'You', 'Ġare', 'Ġa', 'Ġhelpful', 'Ġassistant', '.', '<|im_end|>', 'Ċ', '<|im_start|>', 'user', 'Ċ', 'What']
--------------------------------------------------
Temperature        : 0.7
Max output tokens  : 120
Tokens generated   : 120
--------------------------------------------------
LLM RESPONSE:

Machine learning is a subfield of artificial intelligence that involves the development of algorithms and statistical models that enable computers to learn from data and improve their performance on a specific task over time without being explicitly programmed. It can be thought of as a form of "learning" where the computer "learns" to recognize patterns

In [2]:
# ============================================================
# EXPT 9 - Transformer-based LLM Chatbot using Gradio
# PART B: IMPLEMENTING CHATBOT FUNCTIONALITY
# Name    : Divyadharshini U G
# Roll No : 24BAD023
# ============================================================

import time
import torch

print("Name    : Divyadharshini U G")
print("Roll No : 24BAD023")
print("-" * 50)

SYSTEM_PROMPT = "You are a helpful and concise assistant."

# 1. Chatbot function: accepts a query + history, returns the response
def generate_response(user_input, history=None, temperature=0.7, max_new_tokens=150):
    if history is None:
        history = []

    # Construct the prompt: system message + past turns + new user query
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    messages += history
    messages.append({"role": "user", "content": user_input})

    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

# 2. Chat function that maintains conversation history
conversation_history = []

def chat(user_input, temperature=0.7, max_new_tokens=150):
    start = time.time()
    reply = generate_response(user_input, conversation_history, temperature, max_new_tokens)
    elapsed = time.time() - start

    # Store the turn in history
    conversation_history.append({"role": "user", "content": user_input})
    conversation_history.append({"role": "assistant", "content": reply})

    print(f"User    : {user_input}")
    print(f"Chatbot : {reply}")
    print(f"(Response time: {elapsed:.2f} s)\n")
    return reply

# 3. Test the chatbot with different queries
print("TEST 1: Multi-turn conversation (history ON)")
print("=" * 50)
chat("My name is Divya. Please remember it.")
chat("What is my name?")                       # needs history to answer
chat("Explain what a Transformer model is in two sentences.")
chat("Write a Python function to add two numbers.")

# 4. Same follow-up question WITHOUT history, to show why history matters
print("TEST 2: Same follow-up question (history OFF)")
print("=" * 50)
start = time.time()
no_history_reply = generate_response("What is my name?", history=[])
print("User    : What is my name?")
print("Chatbot :", no_history_reply)
print(f"(Response time: {time.time() - start:.2f} s)")

print("\n" + "-" * 50)
print("Total messages stored in history:", len(conversation_history))

Name    : Divyadharshini U G
Roll No : 24BAD023
--------------------------------------------------
TEST 1: Multi-turn conversation (history ON)
User    : My name is Divya. Please remember it.
Chatbot : Hello Divya! It's Divya again. Is there anything I can assist you with today?
(Response time: 15.50 s)

User    : What is my name?
Chatbot : Your name is Divya.
(Response time: 6.97 s)

User    : Explain what a Transformer model is in two sentences.
Chatbot : A Transformer model is a type of artificial intelligence model that uses self-attention mechanisms to process sequential data, allowing for the modeling of complex relationships between words and concepts.
(Response time: 14.15 s)

User    : Write a Python function to add two numbers.
Chatbot : Here's an example Python function that takes two arguments, adds them together, and returns the result:

```python
def add_numbers(num1, num2):
    return num1 + num2
```

To use this function, simply call it and pass in your desired values a

In [3]:
# ============================================================
# EXPT 9 - Transformer-based LLM Chatbot using Gradio
# PART C: DEVELOPING AN INTERACTIVE INTERFACE
# Name    : Divyadharshini U G
# Roll No : 24BAD023
# ============================================================

import gradio as gr

print("Name    : Divyadharshini U G")
print("Roll No : 24BAD023")
print("Gradio version:", gr.__version__)
print("-" * 50)

# Helper: convert Gradio's history into the format our model expects
def to_messages(history):
    messages = []
    for turn in history:
        if isinstance(turn, dict):                       # messages format
            content = turn["content"]
            if isinstance(content, list):                # content blocks -> plain text
                content = " ".join(c.get("text", "") for c in content if isinstance(c, dict))
            messages.append({"role": turn["role"], "content": content})
        else:                                            # [user, bot] pair format
            user_msg, bot_msg = turn
            messages.append({"role": "user", "content": user_msg})
            if bot_msg:
                messages.append({"role": "assistant", "content": bot_msg})
    return messages

# 1. Connect the interface to the LLM response-generation function (from Part B)
def chatbot_fn(message, history, temperature, max_tokens):
    past = to_messages(history)
    return generate_response(
        message,
        history=past,
        temperature=temperature,
        max_new_tokens=int(max_tokens),
    )

# 2. Build the chat-style interface with a text box, submit button and controls
demo = gr.ChatInterface(
    fn=chatbot_fn,
    additional_inputs=[
        gr.Slider(0.1, 1.5, value=0.7, step=0.1, label="Temperature"),
        gr.Slider(20, 300, value=150, step=10, label="Max output tokens"),
    ],
    title="Transformer LLM Chatbot (Qwen2.5-0.5B-Instruct)",
    description="Expt 9 | Name: Divyadharshini U G | Roll No: 24BAD023",
    examples=[
        ["What is a Transformer model?"],
        ["Explain overfitting in simple words."],
        ["Write a Python function to find the factorial of a number."],
    ],
)

# 3. Run the application (share=True gives a public link you can open in a browser)
demo.launch(share=True, debug=False)

Name    : Divyadharshini U G
Roll No : 24BAD023
Gradio version: 6.30.0
--------------------------------------------------
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3bf7ada8b9ee662561.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [4]:
# ============================================================
# EXPT 9 - PART D: Create project files for GitHub / Hugging Face Spaces
# Name    : Divyadharshini U G
# Roll No : 24BAD023
# ============================================================

import os

print("Name    : Divyadharshini U G")
print("Roll No : 24BAD023")
print("-" * 50)

os.makedirs("llm_chatbot", exist_ok=True)

# ---------- app.py ----------
app_code = '''# Expt 9 - Transformer-based LLM Chatbot
# Name: Divyadharshini U G | Roll No: 24BAD023

import torch
import gradio as gr
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
SYSTEM_PROMPT = "You are a helpful and concise assistant."
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
).to(device)
model.eval()


def to_messages(history):
    messages = []
    for turn in history:
        if isinstance(turn, dict):
            content = turn["content"]
            if isinstance(content, list):
                content = " ".join(c.get("text", "") for c in content if isinstance(c, dict))
            messages.append({"role": turn["role"], "content": content})
        else:
            user_msg, bot_msg = turn
            messages.append({"role": "user", "content": user_msg})
            if bot_msg:
                messages.append({"role": "assistant", "content": bot_msg})
    return messages


def generate_response(user_input, history, temperature, max_tokens):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    messages += to_messages(history)
    messages.append({"role": "user", "content": user_input})

    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=int(max_tokens),
            temperature=temperature,
            do_sample=True,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


demo = gr.ChatInterface(
    fn=generate_response,
    additional_inputs=[
        gr.Slider(0.1, 1.5, value=0.7, step=0.1, label="Temperature"),
        gr.Slider(20, 300, value=150, step=10, label="Max output tokens"),
    ],
    title="Transformer LLM Chatbot (Qwen2.5-0.5B-Instruct)",
    description="Expt 9 | Name: Divyadharshini U G | Roll No: 24BAD023",
)

if __name__ == "__main__":
    demo.launch()
'''

# ---------- requirements.txt ----------
requirements = '''transformers
torch
accelerate
gradio
'''

# ---------- README.md (also used as the Hugging Face Space config) ----------
readme = '''---
title: Transformer LLM Chatbot
emoji: 💬
colorFrom: blue
colorTo: purple
sdk: gradio
app_file: app.py
pinned: false
---

# Transformer-based LLM Chatbot (Gradio)

**Expt 9** | Name: Divyadharshini U G | Roll No: 24BAD023

A chatbot built with the pre-trained Qwen2.5-0.5B-Instruct model from Hugging Face
Transformers, with a Gradio chat interface, conversation history, and sliders for
temperature and maximum output length.

## Run locally
    pip install -r requirements.txt
    python app.py

## Limitations
- Can hallucinate (confident but wrong answers)
- Response latency grows with output length, slower on CPU
- Small model: weaker reasoning than large LLMs
- Limited context window, so long chats lose early turns
'''

for name, content in [("app.py", app_code), ("requirements.txt", requirements), ("README.md", readme)]:
    with open(f"llm_chatbot/{name}", "w", encoding="utf-8") as f:
        f.write(content)

print("Project folder structure:")
for name in sorted(os.listdir("llm_chatbot")):
    size = os.path.getsize(f"llm_chatbot/{name}")
    print(f"  llm_chatbot/{name}  ({size} bytes)")

print("\nContents of requirements.txt:")
print(open("llm_chatbot/requirements.txt").read())

Name    : Divyadharshini U G
Roll No : 24BAD023
--------------------------------------------------
Project folder structure:
  llm_chatbot/README.md  (730 bytes)
  llm_chatbot/app.py  (2356 bytes)
  llm_chatbot/requirements.txt  (37 bytes)

Contents of requirements.txt:
transformers
torch
accelerate
gradio

